# Лабораторная работа 7. Обработка данных из файла. Использование коллекций

ФИО студента: Петров Данила Максимович

Группа студента: ИНБ-б-о-26-2

Вариант: 14

## Задание

В файле `p7_data_14.csv` содержатся данные о товарах: `ID` – уникальный ключ записи; `Name` – наименование; `Category` – категория (4 шт.: A, B, C, D); `Count` – количество; `Price` – цена в долларах; `OutOfDate` – признак просрочки; `WithDiscount` – признак наличия скидки.

Необходимо ответить на следующие вопросы:
1. Средняя цена товаров для просроченных и непросроченных товаров.
2. Суммарный объем в валюте товаров без категории.
3. Самый дорогой товар в каждой категории.
4. Количество наименований товаров без скидки.

## Анализ задания, получение данных

Файл p7_data_14.csv расположен в репозитории. Выполним копирование файла в локальное хранилище Google Colab, используя следующую команду:

In [ ]:
!wget -O p7_data_14.csv https://raw.githubusercontent.com/ReNZzyi/laboratory-and-practical-work-programming-/main/Data/p7_data_14.csv

Можно скопировать и поместить файл с данными любым удобным способом

Реализуем чтение всего файла построчно. Используем метод `open()`, а затем будем обходить файл как коллекцию с использованием цикла `for line in F`. Текстовый файл - это коллекция его строк.
В файле много строк, поэтому будем выводить только первые `n_to_print` строк, но обрабатывать будем все.

In [1]:
F = open('p7_data_14.csv', encoding='utf-8')
n_to_print = 10
for line in F:
  n_to_print -= 1
  if n_to_print > 0:
    print(line, end='')
F.close()

ID,Name,Category,Count,Price,OutOfDate,WithDiscount
1,"Fish - Atlantic Salmon, Cold",D,16,$60.49,true,true
2,Sauce - Cranberry,B,26,$19.52,true,true
3,Pan Grease,D,22,$20.87,,true
4,Coffee Guatemala Dark,A,10,$31.81,,true
5,Wine - Carmenere Casillero Del,,22,$75.15,false,
6,"Olives - Black, Pitted",,5,$95.04,,
7,Tea Leaves - Oolong,B,17,$37.78,,
8,Wine - Prem Select Charddonany,A,7,$26.36,false,false


Видно, что первая строка содержит заголовки столбцов, а не данные. Также видно две особенности файла:
* некоторые наименования заключены в кавычки и **содержат запятую внутри** (например `"Fish - Atlantic Salmon, Cold"`), поэтому простой `split(',')` разрежет такую строку неправильно;
* часть полей пустая: нет категории, не указан признак просрочки или скидки.

Проверим, что произойдёт при использовании `split(',')`:

In [2]:
line = '1,"Fish - Atlantic Salmon, Cold",D,16,$60.49,true,true'
data = line.split(',')
print(data)
print('Количество элементов:', len(data), '- а столбцов всего 7')

['1', '"Fish - Atlantic Salmon', ' Cold"', 'D', '16', '$60.49', 'true', 'true']
Количество элементов: 8 - а столбцов всего 7


Строка разделилась на 8 частей вместо 7, и все значения после наименования сместились. Поэтому для разбора строк воспользуемся стандартным модулем `csv`: функция `csv.reader()` правильно учитывает кавычки и возвращает для каждой строки файла список её полей.

In [3]:
import csv

F = open('p7_data_14.csv', encoding='utf-8')
reader = csv.reader(F)
header = next(reader)   # первая строка - заголовки
print(header)
print('-' * 20)
n_to_print = 10
for data in reader:
  # обрабатываем все строки ...
  n_to_print -= 1
  # ... а печатаем только первые n_to_print
  if n_to_print > 0:
    print(data)
F.close()

['ID', 'Name', 'Category', 'Count', 'Price', 'OutOfDate', 'WithDiscount']
--------------------
['1', 'Fish - Atlantic Salmon, Cold', 'D', '16', '$60.49', 'true', 'true']
['2', 'Sauce - Cranberry', 'B', '26', '$19.52', 'true', 'true']
['3', 'Pan Grease', 'D', '22', '$20.87', '', 'true']
['4', 'Coffee Guatemala Dark', 'A', '10', '$31.81', '', 'true']
['5', 'Wine - Carmenere Casillero Del', '', '22', '$75.15', 'false', '']
['6', 'Olives - Black, Pitted', '', '5', '$95.04', '', '']
['7', 'Tea Leaves - Oolong', 'B', '17', '$37.78', '', '']
['8', 'Wine - Prem Select Charddonany', 'A', '7', '$26.36', 'false', 'false']
['9', 'Corn Meal', 'C', '15', '$95.68', 'false', 'false']


Теперь сформируем структуру данных, максимально полно описывающую предметную область: список словарей `all_goods`, в котором каждый товар представлен словарём. Например, первая строка файла будет выглядеть так:

`example = {'ID': 1, 'Name': 'Fish - Atlantic Salmon, Cold', 'Category': 'D', 'Count': 16, 'Price': 60.49, 'OutOfDate': True, 'WithDiscount': True}`

При заполнении словаря значения приводятся к подходящим типам:
* `Count` – целое число, `Price` – вещественное число (символ `$` отбрасывается);
* `OutOfDate` и `WithDiscount` – логические значения: `True`, если в файле записано `true`. Пустое поле означает, что признак не установлен, поэтому оно считается `False`;
* пустая `Category` означает товар без категории.

In [4]:
F = open('p7_data_14.csv', encoding='utf-8')
reader = csv.reader(F)
header = next(reader)

# список всех товаров
all_goods = []

for data in reader:
  good = {}  # создаем пустой словарь
  for i, key in enumerate(header):  # обходим все заголовки
    good[key] = data[i]  # добавляем в словарь ключ:значение

  # конвертируем значения к подходящим типам
  good['ID'] = int(good['ID'])
  good['Count'] = int(good['Count'])
  good['Price'] = float(good['Price'][1:])  # убираем символ $
  good['OutOfDate'] = good['OutOfDate'] == 'true'
  good['WithDiscount'] = good['WithDiscount'] == 'true'

  all_goods.append(good)  # добавляем товар в список

F.close()  # файл больше не нужен

print('Всего товаров:', len(all_goods))
for good in all_goods[:5]:  # напечатаем первые 5 товаров
  print(good)

Всего товаров: 1000
{'ID': 1, 'Name': 'Fish - Atlantic Salmon, Cold', 'Category': 'D', 'Count': 16, 'Price': 60.49, 'OutOfDate': True, 'WithDiscount': True}
{'ID': 2, 'Name': 'Sauce - Cranberry', 'Category': 'B', 'Count': 26, 'Price': 19.52, 'OutOfDate': True, 'WithDiscount': True}
{'ID': 3, 'Name': 'Pan Grease', 'Category': 'D', 'Count': 22, 'Price': 20.87, 'OutOfDate': False, 'WithDiscount': True}
{'ID': 4, 'Name': 'Coffee Guatemala Dark', 'Category': 'A', 'Count': 10, 'Price': 31.81, 'OutOfDate': False, 'WithDiscount': True}
{'ID': 5, 'Name': 'Wine - Carmenere Casillero Del', 'Category': '', 'Count': 22, 'Price': 75.15, 'OutOfDate': False, 'WithDiscount': False}


## Решение задачи 1

Первая задача: средняя цена товаров для просроченных и непросроченных товаров.

Переберём все товары и будем накапливать сумму цен и количество отдельно для просроченных (`OutOfDate == True`) и непросроченных товаров. Средняя цена = сумма цен / количество товаров.

In [5]:
sum_out, count_out = 0, 0     # просроченные
sum_fresh, count_fresh = 0, 0 # непросроченные

for good in all_goods:
  if good['OutOfDate']:
    sum_out += good['Price']
    count_out += 1
  else:
    sum_fresh += good['Price']
    count_fresh += 1

print('Просроченных товаров: %d, средняя цена = $%.2f' % (count_out, sum_out / count_out))
print('Непросроченных товаров: %d, средняя цена = $%.2f' % (count_fresh, sum_fresh / count_fresh))

Просроченных товаров: 421, средняя цена = $52.25
Непросроченных товаров: 579, средняя цена = $50.66


## Решение задачи 2

Вторая задача: суммарный объём в валюте товаров без категории.

Объём в валюте одного товара = количество × цена. Просуммируем это значение для всех товаров, у которых поле `Category` пустое.

In [6]:
total_no_cat = 0
count_no_cat = 0

for good in all_goods:
  if good['Category'] == '':
    total_no_cat += good['Count'] * good['Price']
    count_no_cat += 1

print('Товаров без категории:', count_no_cat)
print('Суммарный объём в валюте: $%.2f' % total_no_cat)

Товаров без категории: 208
Суммарный объём в валюте: $262807.34


## Решение задачи 3

Третья задача: самый дорогой товар в каждой категории.

Используем словарь `most_expensive`, где ключ – название категории, а значение – самый дорогой товар этой категории, найденный на данный момент. При обходе списка сравниваем цену текущего товара с ценой сохранённого. Товары без категории пропускаем.

In [7]:
most_expensive = {}

for good in all_goods:
  cat = good['Category']
  if cat == '':
    continue  # товары без категории не рассматриваем
  if cat not in most_expensive or good['Price'] > most_expensive[cat]['Price']:
    most_expensive[cat] = good

for cat in sorted(most_expensive):
  good = most_expensive[cat]
  print('Категория %s: %s (ID %d) - $%.2f' % (cat, good['Name'], good['ID'], good['Price']))

Категория A: Bouq All Italian - Primerba (ID 750) - $99.63
Категория B: Country Roll (ID 664) - $99.76
Категория C: Pork Loin Bine - In Frenched (ID 563) - $99.86
Категория D: Juice - Grape, White (ID 155) - $99.63


## Решение задачи 4

Четвёртая задача: количество наименований товаров без скидки.

В файле встречаются повторяющиеся наименования, поэтому для подсчёта **разных** наименований используем множество `set()`: оно хранит только уникальные элементы. Добавим в него наименования всех товаров без скидки (`WithDiscount == False`) и найдём его размер.

In [8]:
names_no_discount = set()
rows_no_discount = 0

for good in all_goods:
  if not good['WithDiscount']:
    names_no_discount.add(good['Name'])
    rows_no_discount += 1

print('Записей о товарах без скидки:', rows_no_discount)
print('Количество наименований товаров без скидки:', len(names_no_discount))

Записей о товарах без скидки: 626
Количество наименований товаров без скидки: 563


В данной работе были использованы операторы `if` (условный оператор), `for ... in` (оператор цикла с обходом коллекции), `continue`. Применялись функции `open()`, `print()`, `len()`, `int()`, `float()`, `sorted()`, `enumerate()`, `next()`, модуль `csv`. В работе использовались типы данных `list`, `dict`, `set`.

# Контрольные вопросы

**1.**

In [9]:
p = [10, 100, 34, 56, 45, 45]
print(sum(p))

290


**2.**

In [10]:
p = [10, 100, 34, 56, 45, 45]
print(len(p))

6


**3.**

In [11]:
p = [10, 100, 35, 34, 56, 45, 45, 21]
print(sum(x for x in p if x % 7 == 0))

112


**4.** Значение `'abc'` нельзя сравнить с числом, поэтому проверяем тип.

In [12]:
p = {4: 10, 12: 100, 35: 3, 34: 56, 56: 11, 45: 'abc'}
print([k for k, v in p.items() if isinstance(v, int) and v > 45])

[12, 34]


**5.**

In [13]:
squares = [x ** 2 for x in range(1, 101)]
print(squares)

[1, 4, 9, 16, 25, 36, 49, 64, 81, 100, 121, 144, 169, 196, 225, 256, 289, 324, 361, 400, 441, 484, 529, 576, 625, 676, 729, 784, 841, 900, 961, 1024, 1089, 1156, 1225, 1296, 1369, 1444, 1521, 1600, 1681, 1764, 1849, 1936, 2025, 2116, 2209, 2304, 2401, 2500, 2601, 2704, 2809, 2916, 3025, 3136, 3249, 3364, 3481, 3600, 3721, 3844, 3969, 4096, 4225, 4356, 4489, 4624, 4761, 4900, 5041, 5184, 5329, 5476, 5625, 5776, 5929, 6084, 6241, 6400, 6561, 6724, 6889, 7056, 7225, 7396, 7569, 7744, 7921, 8100, 8281, 8464, 8649, 8836, 9025, 9216, 9409, 9604, 9801, 10000]


**6.** CSV – текстовый файл: одна строка – одна запись, поля разделены запятой, первая строка обычно содержит заголовки. Значение с запятой внутри заключается в кавычки.

Чтение: открыть файл (`open`) → прочитать заголовки → построчно разделить строки на поля (`split(',')` или `csv.reader`) → привести значения к нужным типам → сохранить в структуру данных → закрыть файл.

**7.** Ключ – `'старт'`, значение – `1472`.

**8.** Добавляет в словарь пару `'id': 54321`. Если ключ `'id'` уже есть, заменяет его значение.

**9.** `ввв`

**10.** Ключ проверяется оператором `in`: `key in d`. Пара целиком – `(key, value) in d.items()`.

**11.** Удаляет из словаря пару с ключом `654`. Если ключа нет, возникает `KeyError`.

**12.** `3`

**13.** Цикл перебирает ключи, поэтому будут выведены `1`, `2`, `3` (каждый с новой строки). Но в записанном виде код не запустится: у `print` нет отступа, а `к` – русская буква.

**14.** `pop(key)` удаляет пару с указанным ключом и возвращает её значение. `popitem()` удаляет последнюю добавленную пару и возвращает кортеж `(ключ, значение)`.

**15.** Представление `dict_items` с парами словаря в виде кортежей `(ключ, значение)`.

**16.** `keys()` возвращает представление `dict_keys` с ключами словаря, `values()` – представление `dict_values` с его значениями.